# 34 — Backtest có kỷ luật

Notebook cuối Track 3. Backtest một chiến lược đà tăng đơn giản, và dùng chính
nó để đo **giá của ba lỗi phổ biến nhất** trong backtest:

| Lỗi | Đo bằng cách |
|---|---|
| **Thiên lệch nhìn trước** ở bộ lọc vũ trụ | chạy lại với bộ lọc dùng dữ liệu cả kỳ |
| **Khớp lệnh cùng thanh tín hiệu** | chạy lại với vào lệnh ngay giá đóng cửa phiên tín hiệu |
| **Bỏ qua chi phí** | quét ngưỡng chi phí từ 0 tới 0,6% |

Chiến lược ở đây không phải điểm chính. **Bộ khung đo lường mới là.**

In [1]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, duong, hom_nay, lui_ngay
from finlens_examples.charts import CHUOI, GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

## 1 · Luật chơi, viết ra trước khi chạy

Viết luật ra **trước** là cách duy nhất chống lại việc chỉnh tham số cho tới
khi đường vốn đẹp. Mỗi con số dưới đây phải có một lý do không phải "vì nó cho
kết quả tốt hơn".

In [2]:
SO_NAM = 5
SO_MA_GIU = 15  # đủ để phân tán, đủ nhỏ để một mã còn ảnh hưởng
CUA_SO_DA = 60  # phiên — đà tăng 3 tháng, khung phổ biến trong tài liệu
NGUONG_GTGD = 5e9  # VND/phiên, tính trên 60 phiên TRƯỚC ngày tái cơ cấu
CHI_PHI_MOT_CHIEU = 0.0018  # 0,18% — phí + thuế bán, xấp xỉ mức bán lẻ Việt Nam
TAN_SUAT_TAI_CO_CAU = "ME"  # cuối mỗi tháng

print(f"Giữ {SO_MA_GIU} mã · đà tăng {CUA_SO_DA} phiên · tái cơ cấu hàng tháng")
print(f"Chi phí một chiều {CHI_PHI_MOT_CHIEU:.2%} → một vòng mua-bán {2 * CHI_PHI_MOT_CHIEU:.2%}")

Giữ 15 mã · đà tăng 60 phiên · tái cơ cấu hàng tháng
Chi phí một chiều 0.18% → một vòng mua-bán 0.36%


## 2 · Dữ liệu

In [3]:
danh_muc = client.meta.symbols(exchange="HOSE", kind="stock")
MA_HOSE = danh_muc["symbol"].tolist()

# Thử cách ngây thơ trước — 405 mã × 5 năm là bao nhiêu dòng?
thu = client.eod.stock.ohlcv(MA_HOSE, start=lui_ngay(HOM_NAY, nam=SO_NAM))
print(f"Một lời gọi: {len(thu):,} dòng · {thu['symbol'].nunique()}/{len(MA_HOSE)} mã")
print(f"truncated = {thu.attrs['finlens']['truncated']}  ·  max_rows của gói = {client.limits()['max_rows']:,}")

C:\Program Files\Python312\Lib\asyncio\events.py:88: DataQualityWarning: Kết quả đã bị cắt ở giới hạn 408425 dòng của gói dịch vụ. Thu hẹp khoảng thời gian, hoặc chia nhỏ danh sách mã.
  self._context.run(self._callback, *self._args)
C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


Một lời gọi: 408,425 dòng · 340/407 mã


truncated = True  ·  max_rows của gói = 100,000


⚠️ **`truncated = True`.** Đây chính là tình huống notebook `00` nói tới, và nó
là chỗ một backtest âm thầm chạy trên dữ liệu thiếu. Không exception nào được
ném; frame trả về trông hoàn toàn bình thường, chỉ là nó thiếu vài chục mã.

Thư viện tự chia lô theo `max_symbols_per_request`, nhưng `max_rows` là một
giới hạn **khác** áp lên kết quả gộp. Cách xử lý: chia theo lô mã, và **kiểm
`truncated` ở từng lô**.

In [4]:
def tai_theo_lo(ma_ds: list[str], *, start: str, so_phien_uoc_tinh: int) -> pd.DataFrame:
    """Tải OHLCV theo lô mã sao cho không lô nào chạm `max_rows`.

    Kích thước lô suy từ `client.limits()["max_rows"]` chứ không viết cứng — gói
    dịch vụ khác thì giới hạn khác. Nếu vẫn bị cắt thì tự chia đôi lô và thử lại.

    ⚠️ `df.attrs` không sống sót qua `pd.concat`, nên đơn vị phải được đọc và
    đối chiếu **trước** khi ghép, rồi gắn lại thủ công.
    """
    max_rows = client.limits()["max_rows"]
    lo_ban_dau = max(10, int(max_rows / so_phien_uoc_tinh * 0.8))  # chừa 20% biên an toàn
    print(f"max_rows={max_rows:,} · ~{so_phien_uoc_tinh} phiên → lô {lo_ban_dau} mã")

    phan, don_vi_chung = [], None

    def lay(lo: list[str]) -> None:
        nonlocal don_vi_chung
        d = client.eod.stock.ohlcv(lo, start=start)
        if d.attrs["finlens"]["truncated"]:
            if len(lo) == 1:
                raise RuntimeError(f"Một mã duy nhất ({lo[0]}) vẫn bị cắt — thu hẹp khoảng thời gian")
            giua = len(lo) // 2
            print(f"  lô {len(lo)} mã bị cắt → chia đôi")
            lay(lo[:giua])
            lay(lo[giua:])
            return
        don_vi = d.attrs["finlens"]["units"]
        if don_vi_chung is None:
            don_vi_chung = don_vi
        elif don_vi != don_vi_chung:
            raise RuntimeError("Hai lô khác đơn vị — không được ghép")
        phan.append(d)

    for i in range(0, len(ma_ds), lo_ban_dau):
        lay(ma_ds[i : i + lo_ban_dau])

    ghep = pd.concat(phan, ignore_index=True)
    ghep.attrs["finlens"] = {"units": don_vi_chung}  # gắn lại thủ công sau concat
    return ghep.sort_values(["symbol", "date"])


# ~250 phiên giao dịch mỗi năm
gia = tai_theo_lo(MA_HOSE, start=lui_ngay(HOM_NAY, nam=SO_NAM), so_phien_uoc_tinh=SO_NAM * 250)
chuan = client.eod.index.ohlcv("VNINDEX", start=lui_ngay(HOM_NAY, nam=SO_NAM)).sort_values("date")

print(f"Tải theo lô: {len(gia):,} dòng · {gia['symbol'].nunique()}/{len(MA_HOSE)} mã")
print(f"  {gia['date'].min():%d/%m/%Y} → {gia['date'].max():%d/%m/%Y}")
print(f"  thêm được {len(gia) - len(thu):,} dòng và {gia['symbol'].nunique() - thu['symbol'].nunique()} mã "
      "so với lời gọi ngây thơ")
print(f"  đơn vị giá: {gia.attrs['finlens']['units']['close']}")

max_rows=100,000 · ~1250 phiên → lô 64 mã


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


Tải theo lô: 489,285 dòng · 405/407 mã
  11/08/2021 → 11/08/2026
  thêm được 80,860 dòng và 65 mã so với lời gọi ngây thơ
  đơn vị giá: kVND


Chuyển sang dạng **rộng** (ngày × mã) — backtest vectorised đọc ma trận nhanh
hơn nhiều so với đọc bảng dạng long.

In [5]:
gia = gia.assign(gtgd=gia["close"] * gia["volume"] * 1_000)

M_GIA = gia.pivot(index="date", columns="symbol", values="close")
M_GTGD = gia.pivot(index="date", columns="symbol", values="gtgd")

print(f"Ma trận giá: {M_GIA.shape[0]} phiên × {M_GIA.shape[1]} mã")
print(f"Ô có dữ liệu: {M_GIA.notna().sum().sum() / M_GIA.size:.1%}  ← mã niêm yết sau sẽ trống ở đầu kỳ")

Ma trận giá: 1247 phiên × 405 mã
Ô có dữ liệu: 96.9%  ← mã niêm yết sau sẽ trống ở đầu kỳ


⚠️ Các ô trống ở đầu kỳ chính là **thiên lệch sống sót đang chờ xảy ra**. Một
backtest lấy danh sách mã của *hôm nay* rồi chạy ngược về quá khứ sẽ chỉ chứa
những doanh nghiệp còn niêm yết tới hôm nay. Cách chống: mọi bộ lọc phải tính
trên **cửa sổ trượt kết thúc trước ngày ra quyết định**, và mã nào chưa có giá
ở ngày đó thì tự động không đủ điều kiện.

## 3 · Bộ khung backtest

Một hàm, ba tham số điều khiển đúng ba lỗi cần đo.

In [6]:
def chay_backtest(
    m_gia: pd.DataFrame,
    m_gtgd: pd.DataFrame,
    *,
    so_ma: int = SO_MA_GIU,
    cua_so: int = CUA_SO_DA,
    nguong_gtgd: float = NGUONG_GTGD,
    chi_phi: float = CHI_PHI_MOT_CHIEU,
    loc_nhin_truoc: bool = False,
    vao_lenh_cung_phien: bool = False,
) -> dict:
    """Backtest đà tăng, tái cơ cấu hàng tháng, phân bổ đều.

    ``loc_nhin_truoc=True`` dùng GTGD bình quân **cả kỳ** để lọc vũ trụ — tức là
    dùng thông tin của tương lai. Để đo giá của lỗi đó.

    ``vao_lenh_cung_phien=True`` tính lợi suất bắt đầu từ chính phiên tín hiệu.
    Trong thực tế bạn chỉ biết tín hiệu **sau khi** phiên đóng cửa.
    """
    ngay_tai_co_cau = m_gia.resample(TAN_SUAT_TAI_CO_CAU).last().index
    ngay_tai_co_cau = [d for d in ngay_tai_co_cau if d in m_gia.index or (m_gia.index <= d).any()]

    ls_ngay = m_gia.pct_change()
    gtgd_ca_ky = m_gtgd.mean()

    danh_muc_theo_ky: dict[pd.Timestamp, list[str]] = {}
    trong_so = pd.DataFrame(0.0, index=m_gia.index, columns=m_gia.columns)
    truoc_do: set[str] = set()
    quay_vong = []

    for i, moc in enumerate(ngay_tai_co_cau):
        # Phiên giao dịch cuối cùng KHÔNG SAU mốc — đây là phiên ta biết mọi thứ
        co = m_gia.index[m_gia.index <= moc]
        if len(co) < cua_so + 2:
            continue
        t = co[-1]

        # ── Bộ lọc vũ trụ ────────────────────────────────────────────────
        if loc_nhin_truoc:
            du_thanh_khoan = gtgd_ca_ky[gtgd_ca_ky >= nguong_gtgd].index  # ⚠️ dùng tương lai
        else:
            cua_so_gtgd = m_gtgd.loc[:t].tail(cua_so)
            bq = cua_so_gtgd.mean()
            du_thanh_khoan = bq[bq >= nguong_gtgd].index

        # ── Tín hiệu: lợi suất `cua_so` phiên tính tới t ──────────────────
        lich_su = m_gia.loc[:t]
        da_tang = lich_su.iloc[-1] / lich_su.iloc[-1 - cua_so] - 1
        du_dieu_kien = da_tang[da_tang.index.isin(du_thanh_khoan)].dropna()
        chon = du_dieu_kien.nlargest(so_ma).index.tolist()
        if not chon:
            continue
        danh_muc_theo_ky[t] = chon

        # ── Kỳ nắm giữ ───────────────────────────────────────────────────
        # Vào lệnh ở phiên SAU t: ta chỉ biết tín hiệu sau khi phiên t đóng cửa.
        vi_tri_t = m_gia.index.get_loc(t)
        bat_dau = vi_tri_t if vao_lenh_cung_phien else vi_tri_t + 1
        ket_thuc = (
            m_gia.index.get_loc(
                m_gia.index[m_gia.index <= ngay_tai_co_cau[i + 1]][-1]
            )
            if i + 1 < len(ngay_tai_co_cau)
            else len(m_gia.index) - 1
        )
        if bat_dau > ket_thuc:
            continue
        trong_so.iloc[bat_dau : ket_thuc + 1, trong_so.columns.get_indexer(chon)] = 1 / len(chon)

        moi = set(chon)
        quay_vong.append(len(moi ^ truoc_do) / (2 * len(moi)) if truoc_do else 1.0)
        truoc_do = moi

    # ── Lợi suất danh mục ────────────────────────────────────────────────
    ls_dm = (trong_so * ls_ngay).sum(axis=1)
    ls_dm[trong_so.sum(axis=1) == 0] = np.nan

    # Chi phí trừ vào ngày trọng số đổi, theo mức thay đổi thực tế
    doi_trong_so = trong_so.diff().abs().sum(axis=1)
    ls_dm = ls_dm - doi_trong_so * chi_phi

    return {
        "loi_suat": ls_dm.dropna(),
        "trong_so": trong_so,
        "danh_muc": danh_muc_theo_ky,
        "quay_vong": float(np.mean(quay_vong)) if quay_vong else np.nan,
    }

Bốn chi tiết trong hàm trên đáng chỉ ra:

1. **`bat_dau = vi_tri_t + 1`** — tín hiệu tính trên giá đóng cửa phiên `t`, nên
   phiên sớm nhất bạn có thể nắm giữ là `t+1`.
2. **Bộ lọc thanh khoản dùng `m_gtgd.loc[:t]`** — chỉ dữ liệu tới `t`, tính lại
   ở *mỗi* lần tái cơ cấu.
3. **`da_tang` dùng `lich_su.iloc[-1]`**, tức giá tại `t`, không phải giá cuối
   bảng.
4. **Chi phí trừ theo `trong_so.diff().abs()`** — mã giữ nguyên qua hai kỳ
   không bị tính phí hai lần.

## 4 · Thước đo

Bốn con số, và một điều đáng nói về con số thứ tư.

In [7]:
def do_luong(ls: pd.Series, ten: str = "") -> dict:
    """CAGR · biến động · Sharpe · sụt giảm sâu nhất · tỷ lệ phiên thắng."""
    ls = ls.dropna()
    if ls.empty:
        return {}
    von = (1 + ls).cumprod()
    so_nam = (ls.index[-1] - ls.index[0]).days / 365.25
    cagr = von.iloc[-1] ** (1 / so_nam) - 1
    bien_dong = ls.std() * np.sqrt(252)
    dinh = von.cummax()
    sut = (von / dinh - 1).min()
    return {
        "chiến lược": ten,
        "CAGR %": round(cagr * 100, 2),
        "Biến động %": round(bien_dong * 100, 2),
        "Sharpe": round(cagr / bien_dong, 2) if bien_dong > 0 else np.nan,
        "Sụt sâu nhất %": round(sut * 100, 2),
        "Phiên thắng %": round((ls > 0).mean() * 100, 1),
        "Số phiên": len(ls),
    }

⚠️ `Sharpe` ở đây dùng **lãi suất phi rủi ro bằng 0**. Trái phiếu chính phủ
Việt Nam kỳ hạn 1 năm không phải 0, nên con số này cao hơn Sharpe thật. Nó vẫn
so sánh được **giữa các chiến lược trong cùng notebook** vì mọi cột đều lệch
cùng một hằng số — nhưng đừng đem nó đi so với Sharpe công bố ở nơi khác.

## 5 · Chạy bản đúng

In [8]:
import time

t0 = time.perf_counter()
dung = chay_backtest(M_GIA, M_GTGD)
print(f"Backtest xong trong {time.perf_counter() - t0:.1f} giây")
print(f"Số kỳ tái cơ cấu: {len(dung['danh_muc'])}")
print(f"Quay vòng danh mục bình quân mỗi kỳ: {dung['quay_vong']:.0%}")

# Chuẩn: VNINDEX, cùng khoảng thời gian
ls_chuan = chuan.set_index("date")["close"].pct_change()
ls_chuan = ls_chuan.loc[dung["loi_suat"].index[0] : dung["loi_suat"].index[-1]]

pd.DataFrame([do_luong(dung["loi_suat"], "Đà tăng 15 mã"), do_luong(ls_chuan, "VNINDEX")])

Backtest xong trong 0.2 giây
Số kỳ tái cơ cấu: 58
Quay vòng danh mục bình quân mỗi kỳ: 55%


,chiến lược,CAGR %,Biến động %,Sharpe,Sụt sâu nhất %,Phiên thắng %,Số phiên
0,Đà tăng 15 mã,-5.70,29.27,-0.19,-62.25,54.8,1169
1,VNINDEX,3.99,19.73,0.20,-40.34,54.9,1169


**Chiến lược thua, và thua đậm.** CAGR âm, biến động cao hơn VNINDEX một nửa,
sụt sâu nhất hơn 60% trong khi chỉ số chỉ sụt 40%.

Đây là kết quả tôi giữ nguyên chứ không đi chỉnh tham số cho tới khi nó đẹp —
và nó có ích hơn một đường vốn dốc lên. Phần còn lại của notebook cho thấy
chính chiến lược thua này **trông có lãi** khi đo bằng phương pháp cẩu thả.

Vì sao đà tăng thuần thua ở đây? Rổ 15 mã đà tăng mạnh nhất toàn sàn thường là
các mã vừa chạy nóng, biến động cao; tái cơ cấu hàng tháng thì mua đúng lúc đà
đã cạn và bán đúng lúc nó bắt đầu lại. Đà tăng hoạt động tốt hơn ở khung dài
hơn, có bộ lọc chất lượng, và có luật cắt lỗ — tức là **thêm ba quyết định
nữa**, mỗi cái lại là một cơ hội để overfit.

In [9]:
von_cl = (1 + dung["loi_suat"]).cumprod() * 100
von_chuan = (1 + ls_chuan.reindex(dung["loi_suat"].index).fillna(0)).cumprod() * 100

so_sanh = pd.concat(
    [
        pd.DataFrame({"date": von_cl.index, "von": von_cl.values, "duong": "Chiến lược đà tăng"}),
        pd.DataFrame({"date": von_chuan.index, "von": von_chuan.values, "duong": "VNINDEX"}),
    ]
)

duong(
    so_sanh,
    x="date",
    y="von",
    theo="duong",
    tieu_de=f"Đường vốn — {SO_NAM} năm, đã trừ chi phí {2 * CHI_PHI_MOT_CHIEU:.2%} mỗi vòng",
    phu_de="Cùng gốc 100 · tái cơ cấu cuối mỗi tháng · vào lệnh phiên kế tiếp",
    nhan_y="giá trị danh mục (gốc = 100)",
)

### Sụt giảm — thứ quyết định bạn có giữ được chiến lược không

CAGR nói bạn kiếm được bao nhiêu *nếu* bạn theo tới cùng. Đường sụt giảm nói
bạn phải chịu đựng những gì để tới đó.

In [10]:
def duong_sut(ls: pd.Series) -> pd.Series:
    von = (1 + ls).cumprod()
    return (von / von.cummax() - 1) * 100


sut = pd.concat(
    [
        pd.DataFrame({"date": dung["loi_suat"].index, "sut": duong_sut(dung["loi_suat"]).values, "duong": "Chiến lược"}),
        pd.DataFrame(
            {
                "date": ls_chuan.reindex(dung["loi_suat"].index).fillna(0).index,
                "sut": duong_sut(ls_chuan.reindex(dung["loi_suat"].index).fillna(0)).values,
                "duong": "VNINDEX",
            }
        ),
    ]
)

duong(
    sut,
    x="date",
    y="sut",
    theo="duong",
    tieu_de="Sụt giảm từ đỉnh",
    phu_de="Khoảng thời gian nằm dưới 0 quan trọng không kém độ sâu",
    nhan_y="%",
    moc_khong=True,
)

## 6 · Đo giá của lỗi 1 — thiên lệch nhìn trước ở bộ lọc vũ trụ

Chạy lại **y hệt**, chỉ đổi một chỗ: bộ lọc thanh khoản dùng GTGD bình quân
của **cả kỳ** thay vì 60 phiên trước ngày ra quyết định. Đây chính là cách
notebook `33` lọc vũ trụ, và cũng là cách hầu hết backtest nghiệp dư lọc.

In [11]:
nhin_truoc = chay_backtest(M_GIA, M_GTGD, loc_nhin_truoc=True)

bang_1 = pd.DataFrame(
    [
        do_luong(dung["loi_suat"], "Đúng — lọc bằng 60 phiên trước"),
        do_luong(nhin_truoc["loi_suat"], "SAI — lọc bằng GTGD cả kỳ"),
    ]
)
bang_1

,chiến lược,CAGR %,Biến động %,Sharpe,Sụt sâu nhất %,Phiên thắng %,Số phiên
0,Đúng — lọc bằng 60 phiên trước,-5.70,29.27,-0.19,-62.25,54.8,1169
1,SAI — lọc bằng GTGD cả kỳ,-4.32,29.19,-0.15,-60.93,55.9,1169


Chênh lệch CAGR ở đây **là món quà mà tương lai tặng cho quá khứ**: bộ lọc cả
kỳ giữ lại những mã *sẽ* có thanh khoản tốt, kể cả khi ở thời điểm ra quyết
định chúng còn chưa ai giao dịch.

Đây là dạng nhìn trước tinh vi nhất, vì nó không nằm trong công thức tín hiệu —
nó nằm ở **danh sách mã bạn cho phép tín hiệu chạy trên đó**.

## 7 · Đo giá của lỗi 2 — khớp lệnh cùng phiên tín hiệu

In [12]:
cung_phien = chay_backtest(M_GIA, M_GTGD, vao_lenh_cung_phien=True)

bang_2 = pd.DataFrame(
    [
        do_luong(dung["loi_suat"], "Đúng — vào lệnh phiên t+1"),
        do_luong(cung_phien["loi_suat"], "SAI — vào lệnh ngay phiên t"),
    ]
)
bang_2

,chiến lược,CAGR %,Biến động %,Sharpe,Sụt sâu nhất %,Phiên thắng %,Số phiên
0,Đúng — vào lệnh phiên t+1,-5.7,29.27,-0.19,-62.25,54.8,1169
1,SAI — vào lệnh ngay phiên t,-1.3,29.58,-0.04,-59.09,55.2,1170


Tín hiệu tính từ **giá đóng cửa** phiên `t`. Nếu bạn tính lợi suất bắt đầu từ
chính phiên `t`, bạn đang giả định mình mua được ở giá mở cửa của một phiên mà
tín hiệu chỉ xuất hiện lúc nó đóng cửa.

Lỗi này thường bị coi là "chỉ lệch một phiên". Bảng trên cho biết một phiên đó
đáng bao nhiêu — và với chiến lược đà tăng, nó thường đáng rất nhiều, vì phiên
tín hiệu chính là phiên giá chạy mạnh nhất.

## 8 · Đo giá của lỗi 3 — bỏ qua chi phí

In [13]:
quet = []
for cp in [0.0, 0.0005, 0.001, 0.0018, 0.0025, 0.003]:
    kq = chay_backtest(M_GIA, M_GTGD, chi_phi=cp)
    d = do_luong(kq["loi_suat"], f"{cp:.2%}")
    d["chi phí một chiều"] = f"{cp:.2%}"
    quet.append(d)

nhay_cam = pd.DataFrame(quet)[["chi phí một chiều", "CAGR %", "Sharpe", "Sụt sâu nhất %"]]
nhay_cam

,chi phí một chiều,CAGR %,Sharpe,Sụt sâu nhất %
0,0.00%,-3.44,-0.12,-61.21
1,0.05%,-4.07,-0.14,-61.50
2,0.10%,-4.70,-0.16,-61.79
3,0.18%,-5.70,-0.19,-62.25
4,0.25%,-6.57,-0.22,-62.64
5,0.30%,-7.18,-0.25,-62.92


In [14]:
ve_cp = nhay_cam.assign(cp=lambda d: d["chi phí một chiều"])
bar_ngang(
    ve_cp,
    nhan="cp",
    gia_tri="CAGR %",
    tieu_de="CAGR theo mức chi phí giao dịch",
    phu_de=f"Tái cơ cấu hàng tháng, quay vòng bình quân {dung['quay_vong']:.0%} mỗi kỳ",
    nhan_x="CAGR %",
    dinh_dang_nhan="{:+.2f}%",
)

Độ dốc của biểu đồ này là **quay vòng nhân với chi phí**. Chiến lược quay vòng
càng cao thì càng nhạy — và một chiến lược tái cơ cấu hàng tuần sẽ dốc gấp bốn
lần cái này.

Đây là lý do "backtest không tính phí" không phải một phiên bản hơi lạc quan
của sự thật. Với chiến lược quay vòng cao, nó là một chiến lược **khác**.

## 9 · Ba lỗi cộng lại

Câu hỏi cuối: nếu mắc cả ba cùng lúc — đúng như một backtest cẩu thả điển hình —
thì đường vốn đẹp thêm bao nhiêu?

In [15]:
tat_ca_loi = chay_backtest(M_GIA, M_GTGD, loc_nhin_truoc=True, vao_lenh_cung_phien=True, chi_phi=0.0)

tong_hop = pd.DataFrame(
    [
        do_luong(dung["loi_suat"], "Bản đúng"),
        do_luong(tat_ca_loi["loi_suat"], "Cả ba lỗi cùng lúc"),
        do_luong(ls_chuan, "VNINDEX"),
    ]
)
tong_hop

,chiến lược,CAGR %,Biến động %,Sharpe,Sụt sâu nhất %,Phiên thắng %,Số phiên
0,Bản đúng,-5.70,29.27,-0.19,-62.25,54.8,1169
1,Cả ba lỗi cùng lúc,3.04,29.49,0.10,-57.64,56.5,1170
2,VNINDEX,3.99,19.73,0.20,-40.34,54.9,1169


In [16]:
von_loi = (1 + tat_ca_loi["loi_suat"]).cumprod() * 100

ba_duong = pd.concat(
    [
        pd.DataFrame({"date": von_cl.index, "von": von_cl.values, "duong": "Bản đúng"}),
        pd.DataFrame({"date": von_loi.index, "von": von_loi.values, "duong": "Cả ba lỗi cùng lúc"}),
        pd.DataFrame({"date": von_chuan.index, "von": von_chuan.values, "duong": "VNINDEX"}),
    ]
)

duong(
    ba_duong,
    x="date",
    y="von",
    theo="duong",
    tieu_de="Cùng một chiến lược, ba cách đo",
    phu_de="Khoảng cách giữa hai đường đầu là toàn bộ phần 'lợi nhuận' do phương pháp sinh ra",
    nhan_y="giá trị danh mục (gốc = 100)",
)

### Đây là toàn bộ luận điểm của notebook này

Hai đường trên đến từ **cùng một chiến lược, cùng một dữ liệu, cùng một khoảng
thời gian**. Chúng chỉ khác nhau ở ba lựa chọn phương pháp — và ba lựa chọn đó
biến một chiến lược lỗ hai chữ số phần trăm mỗi năm thành một chiến lược có
CAGR dương, Sharpe dương, trông hoàn toàn có thể trình bày cho người khác.

Không lựa chọn nào trong ba cái đó trông giống gian lận khi bạn đang viết code.
"Lọc mã thanh khoản" nghe hợp lý. "Vào lệnh ở giá đóng cửa" nghe hợp lý.
"Tạm bỏ phí cho đơn giản" nghe hợp lý. Cả ba đều là những dòng code một người
cẩn thận vẫn viết ra, rồi quên mất.

**Cách phòng duy nhất là đo.** Mỗi khi thêm một giả định vào backtest, chạy lại
cả bản có và bản không, rồi nhìn khoảng cách. Nếu khoảng cách lớn hơn phần lợi
nhuận bạn đang tuyên bố thì cái bạn đang đo là phương pháp, không phải chiến
lược.

## 10 · Danh mục hiện tại

Nếu chạy chiến lược này thật, đây là những mã nó đang giữ.

In [17]:
ky_cuoi = max(dung["danh_muc"])
dang_giu = dung["danh_muc"][ky_cuoi]

print(f"Kỳ tái cơ cấu gần nhất: {ky_cuoi:%d/%m/%Y}")
print(f"Nắm giữ {len(dang_giu)} mã, phân bổ đều {1 / len(dang_giu):.1%} mỗi mã\n")

bang_giu = (
    danh_muc[danh_muc["symbol"].isin(dang_giu)][["symbol", "short_name", "icb_name2"]]
    .assign(
        da_tang_pct=lambda d: [
            round((M_GIA.loc[:ky_cuoi, s].iloc[-1] / M_GIA.loc[:ky_cuoi, s].iloc[-1 - CUA_SO_DA] - 1) * 100, 1)
            for s in d["symbol"]
        ]
    )
    .sort_values("da_tang_pct", ascending=False)
    .reset_index(drop=True)
)
bang_giu

Kỳ tái cơ cấu gần nhất: 11/08/2026
Nắm giữ 15 mã, phân bổ đều 6.7% mỗi mã



,symbol,short_name,icb_name2,da_tang_pct
0,VVS,Đầu tư Phát triển Máy Việt Nam,Ô tô và phụ tùng,48.4
1,PET,Dịch vụ Tổng hợp Dầu khí,Bán lẻ,22.9
2,KLB,KienlongBank,Ngân hàng,16.7
3,FRT,Bán lẻ FPT,Bán lẻ,15.9
4,ACB,ACB,Ngân hàng,15.1
5,DHC,Đông Hải Bến Tre,Tài nguyên Cơ bản,13.1
6,MSB,MSB Bank,Ngân hàng,12.8
7,SSB,SeABank,Ngân hàng,12.1
8,CTF,City Auto,Ô tô và phụ tùng,11.9
9,OCB,Ngân hàng Phương Đông,Ngân hàng,11.3


## Những gì backtest này VẪN chưa tính

Trung thực về giới hạn quan trọng hơn con số CAGR:

| Chưa tính | Ảnh hưởng |
|---|---|
| **Trượt giá** | 15 mã mỗi tháng, lệnh lớn thì giá khớp xấu hơn giá đóng cửa |
| **Biên độ và trần sàn** | mã trần thì không mua được ở giá trần |
| **Cổ tức tiền mặt** | giá đã điều chỉnh nên phần này *có* trong lợi suất, nhưng thuế cổ tức thì không |
| **T+2** | tiền bán chưa về ngay; danh mục thật có ma sát mà mô hình này không có |
| **Một chế độ thị trường** | 5 năm là một mẫu, không phải một phân phối |
| **Chỉ HOSE** | không có HNX, UPCOM |
| **Mã huỷ niêm yết** | `meta.symbols()` chỉ liệt kê mã còn niêm yết hôm nay — phần thiên lệch sống sót còn lại mà bộ lọc trượt không chữa được |

Và điều lớn nhất: **notebook này thử đúng một bộ tham số**. Nếu tôi thử ba mươi
bộ rồi trình bày bộ đẹp nhất, con số sẽ cao hơn nhiều và ý nghĩa sẽ thấp hơn
nhiều. Đó là *overfitting*, và nó không để lại dấu vết nào trong kết quả cuối.

## Tổng kết

| Nguyên tắc | Thực hiện thế nào |
|---|---|
| Không nhìn trước ở tín hiệu | mọi phép tính chỉ dùng `loc[:t]` |
| Không nhìn trước ở **vũ trụ** | bộ lọc tính lại ở mỗi kỳ, trên cửa sổ trượt |
| Vào lệnh sau khi biết tín hiệu | `bat_dau = vi_tri_t + 1` |
| Chi phí theo quay vòng thật | `trong_so.diff().abs()`, không phải phí cố định |
| Luôn có chuẩn so sánh | VNINDEX trên đúng khoảng thời gian đó |
| Công bố cả sụt giảm | CAGR một mình là nửa câu chuyện |

---

**Track 3 hết.** Tiếp theo là Track 4:
[`41_intraday_order_flow.ipynb`](../04-intraday-phai-sinh-vi-mo/41_intraday_order_flow.ipynb) — dữ liệu tick,
ba giá trị của `side`, và ba đại lượng "chủ động" không thay thế được nhau.